#Importing the Libraries

In [41]:
import pandas as pd;
import numpy as np;

#Importing the Dataset

In [42]:
dataset = pd.read_csv('Electricity_Bill_Prediction.csv');

##Splitting the Dataset into the Training Set and Test Set

In [43]:
from sklearn.model_selection import train_test_split;

x_train , x_test , y_train , y_test = train_test_split(dataset.iloc[:,:-1], dataset.iloc[:,-1], test_size=0.2, random_state=42);

In [44]:
len(x_train.columns)

12

In [45]:
x_train.describe()

,Household_Size,Rooms,AC_Units,Fan_Count,Light_Count,Refrigerator,TV_Count,Washing_Machine,Other_Appliances,Temperature_C
count,800.000000,800.000000,800.00000,800.000000,800.000000,800.000000,800.00000,800.000000,800.000000,800.000000
mean,4.936250,4.102500,0.98625,6.867500,9.742500,0.921250,1.13500,0.448750,3.095000,28.435125
std,1.815573,1.792627,0.90254,1.899857,3.728414,0.269517,1.04673,0.497678,1.733416,1.542637
min,2.000000,1.000000,0.00000,3.000000,3.000000,0.000000,0.00000,0.000000,0.000000,24.000000
25%,4.000000,3.000000,0.00000,5.000000,7.000000,1.000000,0.00000,0.000000,2.000000,27.400000
50%,5.000000,4.000000,1.00000,7.000000,10.000000,1.000000,1.00000,0.000000,3.000000,28.400000
75%,6.000000,5.000000,2.00000,8.000000,13.000000,1.000000,2.00000,1.000000,4.000000,29.500000
max,9.000000,7.000000,4.00000,10.000000,17.000000,1.000000,4.00000,1.000000,9.000000,33.300000


In [46]:
x_train['Area_Type'].value_counts(normalize=True)*100

,proportion
Area_Type,
Urban,54.0
Semi-Urban,30.0
Rural,16.0


In [47]:
x_train['Tariff_Category'].value_counts(normalize=True)*100

,proportion
Tariff_Category,
Residential,88.0
Residential-Lifeline,12.0


#Implementation of XG Boost Regressor Model

## Designing the Sklearn Pipeline

In [48]:
from sklearn.preprocessing import StandardScaler;
from sklearn.compose import ColumnTransformer;
from sklearn.preprocessing import OneHotEncoder;
from sklearn.pipeline import Pipeline

In [49]:
numeric_columns = [
    "Household_Size",
    "Rooms",
    "Fan_Count",
    "Light_Count",
    "TV_Count",
    "Other_Appliances",
    "Temperature_C"
]

categorical_columns = [
    "Area_Type",
    "Tariff_Category"
]

In [50]:
ct = ColumnTransformer(

    transformers=[
        ("Scaler", StandardScaler(), numeric_columns),
        ("OHE", OneHotEncoder(drop='first'), categorical_columns)
    ],
    remainder="passthrough"
);

##XGBoost Regressor

In [51]:
from xgboost import XGBRegressor

In [52]:
regressor=XGBRegressor(max_depth=2,gamma=0.7,n_estimators=50)

###Making the Pipeline

In [53]:
pipe = Pipeline([
    ("ct",ct),
     ("regressor",regressor)
     ]);

##XG Boost Regression Model Training

In [54]:
pipe.fit(x_train,y_train);

##Evaluating the Model Performance

In [55]:
from sklearn.metrics import r2_score;

In [56]:
def evaluate(model, x_train, y_train, x_test, y_test):
    # Predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # R² Scores
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    # Adjusted R² function
    def adjusted_r2(r2, n_samples, n_features):
        return 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)

    adj_r2_train = adjusted_r2(r2_train, len(y_train), x_train.shape[1])
    adj_r2_test = adjusted_r2(r2_test, len(y_test), x_test.shape[1])

    # Print results in %
    print(f"\nR² (X_train vs Y_train) : {r2_train*100:.2f} %")
    print(f"R² (X_test vs Y_test)   : {r2_test*100:.2f} %")
    print(f"Adjusted R² (Train)    : {adj_r2_train*100:.2f} %")
    print(f"Adjusted R² (Test)     : {adj_r2_test*100:.2f} %")

In [57]:
evaluate(pipe,x_train, y_train, x_test, y_test)


R² (X_train vs Y_train) : 94.03 %
R² (X_test vs Y_test)   : 93.09 %
Adjusted R² (Train)    : 93.94 %
Adjusted R² (Test)     : 92.65 %


# Saving the model for deployment via Fast API

In [58]:
import joblib

In [59]:
joblib.dump(pipe, "xgb_pipeline.pkl")

['xgb_pipeline.pkl']